# M3 — Tiền Xử Lý, Gán Nhãn & Trích Xuất Đặc Trưng (Colab + Drive)

Chạy **sau** khi M1 (crawl arXiv + Semantic Scholar) và M2 (parsing TeX/BibTeX) đã hoàn thành.

Quy trình: GitHub branch → Drive → đọc dữ liệu M1/M2 → gán nhãn thủ công + tự động → trích xuất 14 đặc trưng → chia split → xuất Parquet.

**Cần push code Python mới nhất lên GitHub branch trước khi chạy notebook.**

## SET UP — Clone code từ branch GitHub và cài thư viện

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# GitHub repository và branch M3
REPO_URL = "https://github.com/tuandatedu/arXir_project.git"
BRANCH = "M3-matching"
PROJECT = Path("/content/project")

# Clone hoặc fetch + pull branch mới nhất
if not (PROJECT / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(PROJECT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only", "origin", BRANCH], check=True)

# Chạy từ project directory
os.chdir(PROJECT)

# Cài thư viện dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "src/requirements.txt"], check=True)

# Chạy unit tests để kiểm tra code
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_m3_matching.py"], check=True)

print("✓ Repository ready:", PROJECT, "|", BRANCH)

## SET UP — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1 — Cấu hình đường dẫn

In [ ]:
RANGE_ID = 1

In [ ]:
import os
from datetime import datetime, timezone
from pathlib import Path

# ========== CONFIGURATION ==========
# Đặt RANGE_ID bằng số range được assign cho bạn (1-5)
RANGE_ID = 1  # TODO: Thay thành range của bạn

# ========== INPUT PATHS (từ M1/M2) ==========
# M1: References crawled từ Semantic Scholar
M1_REFS_JSON = Path(f"/content/drive/MyDrive/references/range_{RANGE_ID}/tables/references.json")

# M2: BibTeX parsed từ PDF, lưu dưới dạng ZIP
M2_ZIP = Path(f"/content/drive/MyDrive/M2/range_{RANGE_ID}/M2_range_{RANGE_ID}_tables.zip")

# ========== OUTPUT PATHS (M3 results) ==========
# Folder chính M3 (trên Drive MyDrive/M3_processed/range_X/)
MY_M3_DIR = Path(f"/content/drive/MyDrive/M3_processed/range_{RANGE_ID}")

# Thư mục con: tables (dữ liệu), labels (nhãn), prepared_dataset (output)
TABLES_DIR = MY_M3_DIR / "tables"
LABELS_DIR = MY_M3_DIR / "labels"
PREPARED_DATASET_DIR = MY_M3_DIR / "prepared_dataset"

# Tạo thư mục nếu chưa tồn tại
for path in (TABLES_DIR, LABELS_DIR, PREPARED_DATASET_DIR):
    path.mkdir(parents=True, exist_ok=True)

# ========== VALIDATION ==========
# Kiểm tra dữ liệu M1/M2 tồn tại trước khi xử lý
assert M1_REFS_JSON.exists(), f"Không tìm thấy {M1_REFS_JSON}"
assert M2_ZIP.exists(), f"Không tìm thấy {M2_ZIP}"

# ========== RUN ID ==========
# Đánh dấu lần chạy (dùng để track pipeline)
RUN_ID = os.environ.get("M3_RUN_ID", datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))

# ========== OVERRIDE GLOBAL CONFIG ==========
# Override config toàn cộng để pipeline ghi vào folder M3 đúng
from src.common import config
import src.common.metrics as metrics_module
config.WORKING_DIR = MY_M3_DIR
config.TABLES_DIR = TABLES_DIR
metrics_module.TABLES_DIR = TABLES_DIR

print(f"Range: {RANGE_ID} | RUN_ID: {RUN_ID}")
print(f"M1 from: {M1_REFS_JSON}")
print(f"M2 from: {M2_ZIP}")
print(f"M3 to: {MY_M3_DIR}")

## 2 — Nạp dữ liệu (M1 + M2) → JSON

In [ ]:
import json

def load_jsonl(file_path: Path) -> list:
    """Load JSONL file (one JSON object per line)."""
    records = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def save_json(records: list, output_path: Path):
    """Save as JSON array."""
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"✓ {output_path.name}: {len(records):,} records")

# Extract M2 ZIP to temporary location
import zipfile
import tempfile

temp_dir = Path(tempfile.mkdtemp())
with zipfile.ZipFile(M2_ZIP, 'r') as zf:
    zf.extractall(temp_dir)
M2_BIBTEX_JSONL = temp_dir / "bibtex.jsonl"

# Load M1 references
with open(M1_REFS_JSON, "r", encoding="utf-8") as f:
    references = json.load(f)
save_json(references, TABLES_DIR / "references.json")

# Load M2 bibtex from extracted ZIP
bibtex = load_jsonl(M2_BIBTEX_JSONL)
save_json(bibtex, TABLES_DIR / "bibtex.json")

print(f"\n✓ Completed: {len(bibtex):,} BibTeX entries, {len(references):,} References")

## 3 — Gán nhãn thủ công (CSV → JSON)

In [ ]:
import pandas as pd
from src.m3_matching.manual_tool import (
    export_manual_template,
    find_candidate_manual_papers,
    import_manual_csv,
    summarize_team_labels,
)

# ========== FIND CANDIDATE PAPERS ==========
# Tìm các bài báo có ≥20 BibTeX entries để gắn nhãn manual
candidates = find_candidate_manual_papers(
    bibtex_path=TABLES_DIR / "bibtex.json",
    references_path=TABLES_DIR / "references.json",
)
print(f"Tìm thấy {len(candidates)} bài đủ điều kiện")
assert len(candidates) >= 5, f"Chỉ có {len(candidates)} bài (cần ≥5)"

# ========== EXPORT CSV TEMPLATES ==========
# Xuất mẫu CSV cho 5 bài báo đầu tiên
# CSV chứa: citation_key, bib_title, suggested_ref_arxiv_id, suggested_score, is_verified
MANUAL_DIR = LABELS_DIR / "manual"
MANUAL_DIR.mkdir(parents=True, exist_ok=True)
selected_5 = [pid for pid, _, _ in candidates[:5]]

for pid in selected_5:
    export_manual_template(
        pid, MANUAL_DIR / f"{pid}.csv",
        bibtex_path=TABLES_DIR / "bibtex.json",
        references_path=TABLES_DIR / "references.json",
    )

print(f"\n✓ Đã xuất {len(selected_5)} CSV tại: {MANUAL_DIR}")
print(f"   → Download CSV, mở Excel, tick is_verified=YES cho những dòng khớp")
print(f"   → Upload lại CSV lên Drive, rồi chạy cell tiếp theo")

# ========== PREVIEW ==========
# Xem trước mẫu CSV đầu tiên
df_preview = pd.read_csv(MANUAL_DIR / f"{selected_5[0]}.csv")
display(df_preview[["citation_key", "bib_title", "suggested_ref_arxiv_id", "suggested_ref_title", "suggested_score", "groundtruth_arxiv_id", "is_verified"]].head(10))

In [ ]:
# ========== IMPORT MANUAL LABELS ==========
# Chạy cell này SAU KHI đã download + tick + upload CSV lên Drive
# Code sẽ: đọc CSV → lọc is_verified=YES → auto-fill groundtruth_arxiv_id → lưu JSON

for pid in selected_5:
    csv_path = MANUAL_DIR / f"{pid}.csv"
    if csv_path.exists():
        import_manual_csv(csv_path, pid, labels_dir=LABELS_DIR)
        print(f"✓ Imported {pid}")

# ========== VERIFY QUOTA ==========
# Kiểm tra đã đạt minimum quota: ≥5 papers, ≥100 entries
summary = summarize_team_labels(labels_dir=LABELS_DIR)
print(f"\nManual Labeling Progress:")
print(f"  Papers: {summary['n_manual_papers']} / 5 required")
print(f"  Entries: {summary['n_manual_entries']} / 100 required")
print(f"  Status: {summary['quota_status']}")

## 4 — Pipeline M3 (14 features + split)

In [ ]:
from src.m3_matching.prepare_dataset import run_preparation_pipeline

# ========== RUN FULL M3 PIPELINE ==========
# Thực hiện:
# 1. Auto-label ≥10% papers ngoài manual pool
# 2. Kiểm tra quota (manual + auto)
# 3. Chia publication thành train/valid/test split (k=2 nghĩa là nhóm 2 người)
# 4. Tạo cặp (BibTeX, Reference) và tính 14 đặc trưng
# 5. Xuất Parquet files

summary = run_preparation_pipeline(
    tables_dir=TABLES_DIR,
    labels_dir=LABELS_DIR,
    k=2,  # TODO: Thay k = số người nhóm
    seed=42,  # Cố định seed để kết quả reproducible
    demo=False,  # False = chạy trên toàn bộ dữ liệu thực
)

print(f"\n{'='*60}")
print(f"📊 PIPELINE RESULTS")
print(f"{'='*60}")
print(f"\nSplit Distribution:")
for split in ["train", "valid", "test"]:
    info = summary["splits"][split]
    print(f"{split.upper():5s}: {info['n_pairs']:,} pairs (pos={info['n_positives']:,}, neg={info['n_negatives']:,})")

print(f"\nOutput Files:")
for f in summary["exported_files"]:
    p = Path(f)
    if p.exists():
        size_mb = p.stat().st_size / 1024 / 1024
        print(f"  ✓ {p.name}: {size_mb:.1f} MB")
    else:
        print(f"  ✗ {p.name}: MISSING")
print(f"\n{'='*60}")

## 5 — Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ========== LOAD TRAINING DATA ==========
# Đọc train set để vẽ biểu đồ phân bố 3 đặc trưng đại diện
train_df = pd.read_parquet(PREPARED_DATASET_DIR / "train_pairs.parquet")

# ========== PLOT FEATURE DISTRIBUTIONS ==========
# Vẽ 3 biểu đồ so sánh positive (1) vs negative (0) pairs
# Dùng cho Poster A2 để minh chứng 14 features phân biệt tốt
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Title similarity - cặp trùng khớp có score cao
sns.histplot(data=train_df, x="title_token_sort", hue="label", bins=30, kde=True, ax=axes[0], palette={0: "#e74c3c", 1: "#2ecc71"}, common_norm=False, stat="density")
axes[0].set_title("Title Similarity (Token Sort Ratio)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Score (0.0 = khác, 1.0 = trùng)")
axes[0].legend(title="Label", labels=["Negative (khác bài)", "Positive (trùng khớp)"])

# Plot 2: Author overlap - cặp trùng khớp thường có tác giả chung
sns.histplot(data=train_df, x="author_jaccard", hue="label", bins=20, ax=axes[1], palette={0: "#e74c3c", 1: "#2ecc71"}, common_norm=False, stat="density")
axes[1].set_title("Author Overlap (Jaccard)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Jaccard Score (0.0 = khác, 1.0 = trùng)")
axes[1].legend(title="Label", labels=["Negative (khác bài)", "Positive (trùng khớp)"])

# Plot 3: Year difference - cặp trùng khớp chênh lệch năm nhỏ
year_df = train_df[train_df["year_diff"] < 20]
sns.boxplot(data=year_df, x="label", y="year_diff", ax=axes[2], palette={0: "#e74c3c", 1: "#2ecc71"})
axes[2].set_title("Year Difference Distribution", fontsize=13, fontweight="bold")
axes[2].set_xlabel("Label")
axes[2].set_ylabel("Chênh lệch năm tuyệt đối")
axes[2].set_xticklabels(["Negative (khác bài)", "Positive (trùng khớp)"])

plt.tight_layout()
poster_fig_path = PREPARED_DATASET_DIR / "poster_feature_distribution.png"
plt.savefig(poster_fig_path, dpi=300)
plt.show()
print(f"✓ Saved: {poster_fig_path}")
print(f"  → Dùng biểu đồ này cho Poster A2 (Phần Feature Engineering)")

## 7 — Required Metrics (for specification compliance)

In [ ]:
print(f"\n{'='*70}")
print(f"📊 REQUIRED METRICS")
print(f"{'='*70}")

# Parse summary data
manual_papers = summary.get('n_manual_papers', 0)
manual_entries = summary.get('n_manual_entries', 0)
auto_papers = summary.get('n_auto_papers', 0)
total_papers = manual_papers + auto_papers
auto_fraction = auto_papers / max(total_papers - manual_papers, 1) if (total_papers - manual_papers) > 0 else 0

print(f"\n1️⃣  MANUAL LABELING COVERAGE:")
manual_ok = "✅ PASS" if (manual_papers >= 5 and manual_entries >= 100) else "❌ FAIL"
print(f"   Papers labeled manually: {manual_papers} / 5 required {manual_ok}")
print(f"   Entries labeled manually: {manual_entries} / 100 required {manual_ok}")

print(f"\n2️⃣  AUTOMATIC LABELING COVERAGE:")
auto_min = max(int(0.1 * (total_papers - manual_papers)), 2) if (total_papers - manual_papers) > 0 else 2
auto_ok = "✅ PASS" if auto_papers >= auto_min else "❌ FAIL"
print(f"   Papers labeled automatically: {auto_papers} / {auto_min} required (≥10% outside manual) {auto_ok}")

print(f"\n3️⃣  LABEL DISTRIBUTION (all splits combined):")
total_pos = sum(summary["splits"][s].get('n_positives', 0) for s in ["train", "valid", "test"])
total_neg = sum(summary["splits"][s].get('n_negatives', 0) for s in ["train", "valid", "test"])
total_pairs = total_pos + total_neg
pos_ratio = total_pos / max(total_pairs, 1) if total_pairs > 0 else 0
print(f"   Total pairs: {total_pairs:,}")
print(f"   Positive (match): {total_pos:,} ({pos_ratio:.1%})")
print(f"   Negative (no match): {total_neg:,} ({1-pos_ratio:.1%})")
print(f"   Imbalance ratio: 1:{total_neg/max(total_pos, 1):.1f} (expected for entity resolution)")

print(f"\n4️⃣  TRAIN/VALID/TEST SPLIT QUALITY:")
for split_name in ["train", "valid", "test"]:
    split_info = summary["splits"][split_name]
    n_pairs = split_info.get('n_pairs', 0)
    n_pos = split_info.get('n_positives', 0)
    n_neg = split_info.get('n_negatives', 0)
    split_pos_ratio = n_pos / max(n_pairs, 1) if n_pairs > 0 else 0
    print(f"   {split_name.upper():5s}: {n_pairs:,} pairs | {n_pos:,} pos ({split_pos_ratio:.1%}), {n_neg:,} neg")

print(f"\n{'='*70}")
compliance = all([
    manual_papers >= 5,
    manual_entries >= 100,
    auto_papers >= auto_min,
    total_pairs > 0
])
status = "✅ SPECIFICATION COMPLIANT" if compliance else "❌ NON-COMPLIANT"
print(f"Compliance Status: {status}")
print(f"{'='*70}")

## 8 — Deliverables

In [ ]:
print(f"\n{'='*70}")
print(f"MILESTONE 3 — COMPLETE")
print(f"{'='*70}")

print(f"\nLocation: {MY_M3_DIR}")
print(f"\n📊 Structure:")
print(f"  tables/")
print(f"    ├── bibtex.json")
print(f"    └── references.json")
print(f"  labels/")
print(f"    ├── manual/")
print(f"    │   ├── 2310.12345.csv  (5 files - edit in Excel)")
print(f"    │   ├── 2310.12346.csv")
print(f"    │   ├── ... .csv")
print(f"    │   ├── 2310.12345.json (imported)")
print(f"    │   ├── 2310.12346.json")
print(f"    │   └── ... .json")
print(f"    └── automatic/")
print(f"        ├── 2310.12345.json")
print(f"        └── ... .json")
print(f"  prepared_dataset/             (🚀 for Person 2)")
print(f"    ├── train_pairs.parquet")
print(f"    ├── valid_pairs.parquet")
print(f"    ├── test_pairs.parquet")
print(f"    ├── dataset_pairs.parquet")
print(f"    └── poster_feature_distribution.png")

print(f"\n✅ Person 1 deliverables:")
print(f"  [✓] Data cleaning: bibtex.json, references.json")
print(f"  [✓] Manual labeling: 5 CSV + JSON")
print(f"  [✓] Auto labeling: JSON")
print(f"  [✓] Feature engineering: 14 features")
print(f"  [✓] Dataset creation: pairs with label")
print(f"  [✓] Train/valid/test split")

print(f"\n🚀 Person 2 receives:")
print(f"  • prepared_dataset/ folder")
print(f"  • 4 Parquet files + visualization")
print(f"  • Ready for model training")

print(f"\nRUN_ID: {RUN_ID}")
print(f"{'='*70}")